In [1]:
import os, glob, json
from pathlib import Path
import numpy as np
from tokenizers import Tokenizer

DATA = Path("/kaggle/input/datasets/megs132/capstone-1")

for p in sorted(DATA.iterdir()):
    print(f"{p.stat().st_size/1e6:9.1f} MB  {p.name}")

WORK     = "/kaggle/working"
CKPT_DIR = f"{WORK}/ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

TRAIN_BIN, VAL_BIN = DATA / "bpe32k_train.bin", DATA / "bpe32k_val.bin"
META_PATH, TOK_PATH = DATA / "bpe32k_meta.json", DATA / "bpe32k.json"
TEST_PATH = DATA / "test.json"
VAL_PROMPTS_PATH, VAL_REFS_PATH = DATA / "val_prompts.json", DATA / "val_refs.json"

for p in (TRAIN_BIN, VAL_BIN, META_PATH, TOK_PATH, TEST_PATH,
          VAL_PROMPTS_PATH, VAL_REFS_PATH):
    assert p.exists(), f"missing: {p}"

META = json.load(open(META_PATH))
VOCAB_SIZE = META["vocab_size"]
PAD_ID = META["special_token_ids"]["pad"]
EOS_ID = META["special_token_ids"]["eos"]

tok = Tokenizer.from_file(str(TOK_PATH))
train_bin = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_bin   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")

test_data   = json.load(open(TEST_PATH))
VAL_PROMPTS = json.load(open(VAL_PROMPTS_PATH))
VAL_REFS    = json.load(open(VAL_REFS_PATH))

print(f"\nvocab {VOCAB_SIZE:,} | pad {PAD_ID} | eos {EOS_ID}")
print(f"train {len(train_bin):,} tokens | val {len(val_bin):,} tokens")
print(f"test {len(test_data)} | val prompts {len(VAL_PROMPTS)}")
print("\nsample:", tok.decode(train_bin[:60].astype(np.int64).tolist())[:300])

      0.0 MB  __huggingface_repos__.json
      2.3 MB  bpe32k.json
      0.0 MB  bpe32k_meta.json
    434.4 MB  bpe32k_train.bin
      5.1 MB  bpe32k_val.bin
      0.0 MB  corpus_stats.json
      3.0 MB  test.json
    181.6 MB  train_acts.jsonl
    367.1 MB  train_high_courts.jsonl
    296.2 MB  train_supreme_court.jsonl
    183.4 MB  train_tribunals.jsonl
      3.1 MB  val_acts.jsonl
      3.1 MB  val_high_courts.jsonl
      0.9 MB  val_prompts.json
      0.2 MB  val_refs.json
      3.0 MB  val_supreme_court.jsonl
      3.1 MB  val_tribunals.jsonl

vocab 32,000 | pad 0 | eos 3
train 217,201,090 tokens | val 2,560,200 tokens
test 1000 | val prompts 300

sample: Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors inclu


In [2]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DIM, N_HEADS, MAX_SEQ = 768, 12, 2048
BLOCK = 1024
USE_AMP = DEVICE == "cuda"
print(f"device {DEVICE} | vocab {VOCAB_SIZE:,}")

device cuda | vocab 32,000


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x = x.float()                                   # fp32 reduction
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x * rms).to(dtype) * self.weight


class SwiGLU(nn.Module):
    def __init__(self, dim, mult=4, multiple_of=64):
        super().__init__()
        hidden = int(dim * mult * 2 / 3)
        hidden = multiple_of * ((hidden + multiple_of - 1) // multiple_of)

        self.w1 = nn.Linear(dim, hidden, bias=False)    # gate branch
        self.w2 = nn.Linear(dim, hidden, bias=False)    # value branch
        self.w3 = nn.Linear(hidden, dim, bias=False)    # project back

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


# ---------------- tests ----------------

torch.manual_seed(42)
B, T, D = 4, 32, 768
x = torch.randn(B, T, D)

norm = RMSNorm(D)
y = norm(x)
rms = y.pow(2).mean(-1).sqrt()
assert y.shape == x.shape
assert torch.allclose(rms, torch.ones_like(rms), atol=1e-4)
print(f"RMSNorm   ok | out RMS {rms.mean():.6f} | params {sum(p.numel() for p in norm.parameters()):,}")

# fp16 stability: large activations must not overflow
xb = torch.randn(2, 8, D) * 300
assert torch.isfinite(norm(xb.half() if torch.cuda.is_available() else xb)).all()
print("RMSNorm   fp16 overflow test ok")

mlp = SwiGLU(D)
z = mlp(x)
assert z.shape == x.shape
print(f"SwiGLU    ok | hidden {mlp.w1.out_features} | params {sum(p.numel() for p in mlp.parameters()):,}")

RMSNorm   ok | out RMS 1.000000 | params 768
RMSNorm   fp16 overflow test ok
SwiGLU    ok | hidden 2048 | params 4,718,592


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RoPE(nn.Module):
    """Contiguous-half rotary embeddings.
    x = [x1 | x2] -> [x1*cos - x2*sin, x2*cos + x1*sin]
    This convention MUST match the inference path exactly."""

    def __init__(self, head_dim, max_seq):
        super().__init__()
        assert head_dim % 2 == 0, "head_dim must be even for RoPE"
        self.head_dim, self.max_seq = head_dim, max_seq

        inv_freq = 1.0 / (10000 ** (
            torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_seq, dtype=torch.float32), inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, x, pos0=0):
        """x: (B, H, T, head_dim). pos0 is the absolute position of x[..., 0, :],
        used by cached generation where T == 1."""
        dtype = x.dtype
        B, H, T, D = x.shape
        assert D == self.head_dim
        assert pos0 + T <= self.max_seq, f"position {pos0 + T} exceeds max_seq {self.max_seq}"

        half = D // 2
        x = x.float()
        x1, x2 = x[..., :half], x[..., half:]

        cos = self.cos[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)

        return torch.cat([x1 * cos - x2 * sin,
                          x2 * cos + x1 * sin], dim=-1).to(dtype)


class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq, rope=True):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"

        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.qkv      = nn.Linear(dim, 3 * dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope     = RoPE(self.head_dim, max_seq) if rope else None

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)   # (B, H, T, hd)

        if self.rope is not None:
            q, k = self.rope(q), self.rope(k)            # never v

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)


# ============================================================
# TESTS
# ============================================================

torch.manual_seed(42)
B, T, D, H = 2, 16, 768, 12

attn = CausalSelfAttention(D, H, max_seq=2048, rope=False)
x = torch.randn(B, T, D)
y = attn(x)
assert y.shape == x.shape and torch.isfinite(y).all()
print("attention  shape/finite ok")

# causality: perturbing position t must not affect positions < t
t = 7
x1 = torch.randn(B, T, D)
x2 = x1.clone()
x2[:, t, :] = torch.randn(B, D)
with torch.no_grad():
    y1, y2 = attn(x1), attn(x2)

assert torch.allclose(y1[:, :t], y2[:, :t], atol=1e-6), "LEAK: past positions changed"
assert not torch.allclose(y1[:, t:], y2[:, t:], atol=1e-6), "position t had no effect"
print("attention  causality ok")

# --- RoPE ---
hd, max_seq = 64, 2048
rope = RoPE(hd, max_seq)

q = torch.randn(2, 8, 32, hd)
assert torch.allclose(q.norm(dim=-1), rope(q).norm(dim=-1), atol=1e-5), "not a rotation"
print("rope       norm preservation ok")

# relative position: score depends only on (i - j), with DISTINCT q and k
qv, kv = torch.randn(hd), torch.randn(hd)
qt, kt = torch.zeros(1, 1, 16, hd), torch.zeros(1, 1, 16, hd)
qt[0, 0, 5], kt[0, 0, 3]   = qv, kv      # offset 2
qt[0, 0, 12], kt[0, 0, 10] = qv, kv      # offset 2
qr, kr = rope(qt), rope(kt)

d1 = (qr[0, 0, 5]  * kr[0, 0, 3]).sum()
d2 = (qr[0, 0, 12] * kr[0, 0, 10]).sum()
d3 = (qr[0, 0, 12] * kr[0, 0, 3]).sum()   # offset 9 — must differ

assert torch.allclose(d1, d2, atol=1e-5), f"not relative: {d1:.6f} vs {d2:.6f}"
assert not torch.allclose(d1, d3, atol=1e-3), "position has no effect at all"
print(f"rope       relative position ok  (off2 {d1:.4f}={d2:.4f}, off9 {d3:.4f})")

# pos0 offset must equal the same absolute position computed in one pass
full = rope(qt)
piece = rope(qt[:, :, 12:13], pos0=12)
assert torch.allclose(full[:, :, 12:13], piece, atol=1e-5), "pos0 offset broken"
print("rope       pos0 offset ok")

# --- combined ---
rattn = CausalSelfAttention(768, 12, max_seq=2048)
xr = torch.randn(2, 128, 768)
with torch.no_grad():
    yr = rattn(xr)
assert yr.shape == xr.shape and torch.isfinite(yr).all()
print(f"rope-attn  ok | params {sum(p.numel() for p in rattn.parameters()):,}")

attention  shape/finite ok
attention  causality ok
rope       norm preservation ok
rope       relative position ok  (off2 -0.3179=-0.3179, off9 -4.3299)
rope       pos0 offset ok
rope-attn  ok | params 2,359,296


In [9]:
class GatedLinearRecurrence(nn.Module):
    """h_t = a_t * h_{t-1} + i_t * v_t, computed chunkwise in parallel."""

    def __init__(self, dim, conv_kernel=3, chunk_size=16):
        super().__init__()
        self.dim, self.chunk_size, self.conv_kernel = dim, chunk_size, conv_kernel

        self.in_proj = nn.Linear(dim, 4 * dim, bias=False)
        self.dwconv = nn.Conv1d(dim, dim, conv_kernel,
                                padding=conv_kernel - 1, groups=dim, bias=False)
        self.decay_proj = nn.Linear(dim, dim, bias=True)
        self.out_proj   = nn.Linear(dim, dim, bias=False)

    def _scan(self, log_a, u):
        B, T, D = u.shape
        L = self.chunk_size
        pad = (-T) % L
        if pad:
            log_a = F.pad(log_a, (0, 0, 0, pad))
            u     = F.pad(u,     (0, 0, 0, pad))
        n = (T + pad) // L

        log_a = log_a.view(B, n, L, D)
        u     = u.view(B, n, L, D)
        A = log_a.cumsum(2)

        d = A.unsqueeze(3) - A.unsqueeze(2)
        tri = torch.tril(torch.ones(L, L, device=u.device, dtype=torch.bool))
        d = d.masked_fill(~tri[None, None, :, :, None], float("-inf"))
        M = torch.exp(d)

        local = torch.einsum("bnijd,bnjd->bnid", M, u)

        h, outs = u.new_zeros(B, D), []
        decay = torch.exp(A)
        for c in range(n):
            outs.append(local[:, c] + decay[:, c] * h[:, None])
            h = outs[-1][:, -1]
        return torch.cat(outs, 1)[:, :T]

    def forward(self, x):
        B, T, D = x.shape
        c = self.dwconv(x.transpose(1, 2))[..., :T].transpose(1, 2)

        v, i, _, o = self.in_proj(c).chunk(4, dim=-1)
        v, i, o = torch.tanh(v), torch.sigmoid(i), torch.sigmoid(o)

        log_a = -0.01 - 0.99 * torch.sigmoid(self.decay_proj(c).float())
        h = self._scan(log_a, i.float() * v.float())
        return self.out_proj((h * o.float()).to(x.dtype))

In [10]:
class GQASlidingWindowAttention(nn.Module):
    """Grouped-query attention with an optional sliding window.
    Fewer KV heads than Q heads; each layer sees only the last `window_size`
    tokens unless sliding_window=False."""

    def __init__(self, dim, n_heads, n_kv_heads=4, max_seq=2048,
                 window_size=256, sliding_window=True):
        super().__init__()
        assert dim % n_heads == 0 and n_heads % n_kv_heads == 0

        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = dim // n_heads
        self.window_size, self.sliding_window = window_size, sliding_window

        self.q_proj = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

        self.rope = RoPE(self.head_dim, max_seq)

        pos = torch.arange(max_seq)
        d = pos[:, None] - pos[None, :]
        causal = d >= 0
        self.register_buffer("causal_mask", causal, persistent=False)
        self.register_buffer("window_mask", causal & (d < window_size),
                             persistent=False)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads,    self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q, k = self.rope(q), self.rope(k)

        mask = (self.window_mask if self.sliding_window else self.causal_mask)[:T, :T]

        # enable_gqa broadcasts the KV heads across query groups without
        # materialising a full-size K/V, preserving the memory saving
        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=False,
            enable_gqa=(self.n_kv_heads != self.n_heads))

        return self.out_proj(out.transpose(1, 2).reshape(B, T, C))

In [11]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

USE_AMP = DEVICE == "cuda"
C_LAYERS, C_ATTN_EVERY = 12, 3


class GenericBlock(nn.Module):
    """Pre-norm residual block with an injected token mixer."""
    def __init__(self, dim, mixer):
        super().__init__()
        self.norm1, self.mixer = RMSNorm(dim), mixer
        self.norm2, self.mlp   = RMSNorm(dim), SwiGLU(dim)

    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class GenericLM(nn.Module):
    def __init__(self, mixer_factory, vocab_size=VOCAB_SIZE, dim=DIM,
                 n_layers=12, max_seq=MAX_SEQ):
        super().__init__()
        self.vocab_size, self.dim, self.max_seq = vocab_size, dim, max_seq

        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.blocks  = nn.ModuleList(
            [GenericBlock(dim, mixer_factory(i)) for i in range(n_layers)])
        self.norm_f  = RMSNorm(dim)
        self.head    = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight          # tie

        self.apply(self._init_weights)

        std = 0.02 / math.sqrt(2 * n_layers)
        for b in self.blocks:
            nn.init.normal_(b.mixer.out_proj.weight, 0.0, std)
            nn.init.normal_(b.mlp.w3.weight,         0.0, std)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        T = idx.size(1)
        if T > self.max_seq:
            raise ValueError(f"T={T} exceeds max_seq={self.max_seq}")

        x = self.tok_emb(idx)
        for b in self.blocks:
            x = b(x)
        logits = self.head(self.norm_f(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size),
                                   targets.reshape(-1))
        return logits, loss


def count_unique_params(m):
    return sum(p.numel() for p in {id(p): p for p in m.parameters()}.values())


def c_mixer_factory(i):
    """Attention every 3rd layer; gated linear recurrence elsewhere."""
    if (i + 1) % C_ATTN_EVERY == 0:
        return CausalSelfAttention(DIM, N_HEADS, max_seq=MAX_SEQ)
    return GatedLinearRecurrence(DIM, conv_kernel=3, chunk_size=16)


# ============================================================
# CHECKS
# ============================================================

torch.manual_seed(42)
model_c = GenericLM(c_mixer_factory, n_layers=C_LAYERS).to(DEVICE)

# ---- 1. parameter count ----
C_PARAMS = count_unique_params(model_c)
n_attn = sum(1 for i in range(C_LAYERS) if (i + 1) % C_ATTN_EVERY == 0)

print(f"params {C_PARAMS:,} ({C_PARAMS/1e6:.2f}M)")
assert 50_000_000 <= C_PARAMS <= 300_000_000, f"{C_PARAMS:,} outside 50M-300M"
print(f"  {n_attn} attention layers, {C_LAYERS - n_attn} recurrence layers")
print(f"  embedding (tied) {model_c.tok_emb.weight.numel():,}")

# ---- 2. initial loss ----
torch.manual_seed(42)
x = torch.randint(0, VOCAB_SIZE, (2, 512), device=DEVICE)
y = torch.randint(0, VOCAB_SIZE, (2, 512), device=DEVICE)

model_c.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    _, init_loss = model_c(x, y)
init_loss = init_loss.item()

print(f"\ninitial loss {init_loss:.4f} vs ln(vocab) {math.log(VOCAB_SIZE):.4f}")
assert 9.0 < init_loss < 12.0, f"unexpected init loss {init_loss:.4f}"

# ---- 3. causality: perturb a LATE token, earlier outputs must not move ----
torch.manual_seed(123)
xa = torch.randint(0, VOCAB_SIZE, (1, 256), device=DEVICE)
xb = xa.clone()
t = 200
xb[0, t] = (xa[0, t].item() + 1) % VOCAB_SIZE

with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    la, _ = model_c(xa)
    lb, _ = model_c(xb)

before = (la[:, :t].float() - lb[:, :t].float()).abs().max().item()
after  = (la[:, t:].float() - lb[:, t:].float()).abs().max().item()
print(f"\ncausality: max delta before t {before:.3e} | from t {after:.3e}")
assert before < 1e-4, "LEAK: future token affected past positions"
assert after > 1e-3, "perturbation had no effect"

# ---- 4. overfit one real batch ----
model_c.train()
rng = np.random.RandomState(12345)
off = rng.randint(0, len(train_bin) - BLOCK - 1, size=2)

xo = torch.from_numpy(
    np.stack([train_bin[i:i + BLOCK] for i in off]).astype(np.int64)).to(DEVICE)
yo = torch.from_numpy(
    np.stack([train_bin[i + 1:i + 1 + BLOCK] for i in off]).astype(np.int64)).to(DEVICE)

opt = torch.optim.AdamW(model_c.parameters(), lr=1e-3, weight_decay=0.0)
sc  = torch.amp.GradScaler("cuda", enabled=USE_AMP)
hist = []

for step in range(100):
    opt.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
        _, loss = model_c(xo, yo)
    sc.scale(loss).backward()
    sc.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model_c.parameters(), 1.0)
    sc.step(opt); sc.update()
    hist.append(loss.item())
    if step == 0 or (step + 1) % 20 == 0:
        print(f"  overfit {step+1:3d}/100  loss {loss.item():.4f}")

print(f"\noverfit {hist[0]:.4f} -> {hist[-1]:.4f}")
assert hist[-1] < 1.0, f"gradients not flowing: {hist[-1]:.4f}"

del opt, sc, xo, yo
torch.cuda.empty_cache()

print(f"\nMODEL C READY — {C_PARAMS/1e6:.2f}M params "
      f"({100*(C_PARAMS-109_529_856)/109_529_856:+.1f}% vs A)")
print(f"hybrid: recurrence in {C_LAYERS-n_attn} layers, attention every {C_ATTN_EVERY}")

params 118,991,616 (118.99M)
  4 attention layers, 8 recurrence layers
  embedding (tied) 24,576,000

initial loss 10.5312 vs ln(vocab) 10.3735

causality: max delta before t 0.000e+00 | from t 3.365e+00
  overfit   1/100  loss 10.5143
  overfit  20/100  loss 4.7991
  overfit  40/100  loss 1.0254
  overfit  60/100  loss 0.0124
  overfit  80/100  loss 0.0009
  overfit 100/100  loss 0.0004

overfit 10.5143 -> 0.0004

MODEL C READY — 118.99M params (+8.6% vs A)
hybrid: recurrence in 8 layers, attention every 3


In [12]:
import glob, torch
from pathlib import Path

hits = glob.glob("/kaggle/input/**/D_gqa.pt", recursive=True)
assert hits, "phase1-model-d not attached"
CKPT_D = Path(hits[0])

full = glob.glob("/kaggle/input/**/D_gqa_last.pt", recursive=True)
if full:
    print("trained config:", torch.load(full[0], map_location="cpu",
                                        weights_only=False)["config"])

D_LAYERS, D_FULL_EVERY, D_WINDOW, D_KV_HEADS = 12, 3, 256, 4

def d_mixer_factory(i):
    return GQASlidingWindowAttention(
        DIM, N_HEADS, n_kv_heads=D_KV_HEADS, max_seq=MAX_SEQ,
        window_size=D_WINDOW, sliding_window=((i + 1) % D_FULL_EVERY != 0))

model_d = GenericLM(d_mixer_factory, n_layers=D_LAYERS).to(DEVICE)
sd = torch.load(CKPT_D, map_location="cpu")
model_d.load_state_dict({k: v.float() for k, v in sd.items()}, strict=True)
model_d.eval()
print(f"loaded {count_unique_params(model_d)/1e6:.2f}M params")

trained config: {'arch': 'glr-attention hybrid', 'n_layers': 12, 'attn_every': 3, 'chunk_size': 32, 'block': 1024, 'micro_bs': 8, 'accum': 4, 'lr': 0.0006, 'min_lr': 6e-05, 'warmup': 200, 'wd': 0.1, 'max_steps': 6600}
loaded 100.09M params


In [14]:
NEG = -1e4

SPECIAL_TOKEN_IDS = [i for i in (
    tok.token_to_id("<|pad|>"), tok.token_to_id("<|unk|>"),
    tok.token_to_id("<|bos|>"), tok.token_to_id("<|eos|>")) if i is not None]
PAD_ID = int(tok.token_to_id("<|pad|>"))


@torch.no_grad()
def sample_next_token(logits, history, ngram_maps=None,
                      temperature=1.0, top_k=None, top_p=None,
                      repetition_penalty=1.0, repetition_window=128,
                      no_repeat_ngram_size=None):
    logits = logits.float().clone()
    logits[:, SPECIAL_TOKEN_IDS] = NEG

    if repetition_penalty != 1.0 and history.shape[1] > 0:
        recent = history[:, -repetition_window:]
        g = torch.gather(logits, 1, recent)
        g = torch.where(g > 0, g / repetition_penalty, g * repetition_penalty)
        logits.scatter_(1, recent, g)

    n = no_repeat_ngram_size or 0
    if n > 1 and ngram_maps is not None:
        for b in range(logits.shape[0]):
            seq = history[b].tolist()
            if len(seq) >= n - 1:
                banned = ngram_maps[b].get(tuple(seq[-(n - 1):]))
                if banned:
                    logits[b, list(banned)] = NEG

    if temperature <= 0:
        return logits.argmax(-1, keepdim=True)

    logits = logits / temperature

    if top_k:
        k = min(top_k, logits.shape[-1])
        kth = logits.topk(k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, NEG)

    if top_p is not None and top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cum = sl.softmax(-1).cumsum(-1)
        drop = cum > top_p
        drop[:, 1:] = drop[:, :-1].clone()
        drop[:, 0] = False
        logits = logits.masked_fill(
            torch.zeros_like(drop).scatter_(1, si, drop), NEG)

    probs = logits.softmax(-1)
    bad = ~torch.isfinite(probs).all(-1) | (probs.sum(-1) <= 0)
    if bad.any():
        probs[bad] = 0.0
        probs[bad, logits[bad].argmax(-1)] = 1.0
    return torch.multinomial(probs, 1)


def update_ngram_maps(ngram_maps, history, n):
    if n > 1:
        for b in range(history.shape[0]):
            seq = history[b].tolist()
            if len(seq) >= n:
                ngram_maps[b].setdefault(tuple(seq[-n:-1]), set()).add(seq[-1])

In [15]:
def postprocess_predictions(results, test_data, target_words=128):
    preds, short = {}, []

    for key in test_data:
        key = str(key)
        words = tok.decode(results[key]["generated_ids"]).split()
        if len(words) < target_words:
            short.append((key, len(words)))
        preds[key] = {"text": " ".join(words[:target_words])}

    assert set(preds) == {str(k) for k in test_data}, "key set mismatch"

    for key, p in preds.items():
        pt = test_data[key]["text"] if isinstance(test_data[key], dict) else test_data[key]
        assert not p["text"].startswith(pt[:80]), f"echoes prompt: {key}"

    print(f"postprocess | short: {len(short)}/{len(preds)}"
          + (f" -> {short[:5]}" if short else ""))
    return preds, short

In [16]:
import torch
import torch.nn.functional as F


def cached_gqa(mixer, x, cache=None, pos0=0):
    """GQA attention with a window-truncated KV cache.
    Cache stays at n_kv_heads (compact); enable_gqa broadcasts to n_heads.
    Windowed layers never hold more than window_size positions."""
    B, T, C = x.shape
    H, HKV, HD = mixer.n_heads, mixer.n_kv_heads, mixer.head_dim

    q = mixer.q_proj(x).view(B, T, H,   HD).transpose(1, 2)
    k = mixer.k_proj(x).view(B, T, HKV, HD).transpose(1, 2)
    v = mixer.v_proj(x).view(B, T, HKV, HD).transpose(1, 2)

    q = mixer.rope(q, pos0=pos0)
    k = mixer.rope(k, pos0=pos0)

    if cache is not None:
        k = torch.cat([cache[0], k], dim=2)
        v = torch.cat([cache[1], v], dim=2)

    gqa = (HKV != H)

    if T == 1:
        # one query at absolute pos0: windowed layers see only the last W keys,
        # which is exactly the training window. No mask needed.
        if mixer.sliding_window:
            W = mixer.window_size
            ka, va = k[:, :, -W:], v[:, :, -W:]
        else:
            ka, va = k, v

        y = F.scaled_dot_product_attention(q, ka, va, is_causal=False,
                                           enable_gqa=gqa)
    else:
        # prefill: rebuild the same mask used in training, in absolute positions
        K = k.size(2)
        qp = torch.arange(pos0, pos0 + T, device=x.device)
        kp = torch.arange(K - (pos0 + T) + (pos0 + T), device=x.device)
        kp = torch.arange(K, device=x.device) + (pos0 + T - K)
        d = qp[:, None] - kp[None, :]
        mask = (d >= 0) & (d < mixer.window_size) if mixer.sliding_window else (d >= 0)

        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           is_causal=False, enable_gqa=gqa)

    y = mixer.out_proj(y.transpose(1, 2).reshape(B, T, C))

    # truncate the stored cache so windowed layers stay bounded
    if mixer.sliding_window:
        W = mixer.window_size
        return y, (k[:, :, -W:], v[:, :, -W:])
    return y, (k, v)

In [17]:
import torch


@torch.no_grad()
def cached_forward_d(model, idx, states=None, pos0=0):
    x = model.tok_emb(idx)
    new_states = []

    for i, block in enumerate(model.blocks):
        out, cache = cached_gqa(block.mixer, block.norm1(x),
                                None if states is None else states[i], pos0)
        x = x + out
        x = x + block.mlp(block.norm2(x))
        new_states.append(cache)

    x = model.norm_f(x)
    return model.head(x[:, -1:]), new_states


@torch.no_grad()
def prefill_d(model, prompt):
    logits, states = cached_forward_d(model, prompt, states=None, pos0=0)
    return logits, states, prompt.size(1)


@torch.no_grad()
def step_d(model, token, states, pos0):
    """pos0 must be passed explicitly — windowed caches stop growing at
    window_size, so cache length can't be used to infer position."""
    return cached_forward_d(model, token, states=states, pos0=pos0)

In [18]:
import numpy as np
import torch

model_d.eval()

T_TEST, T_PREFILL, N_STEPS = 800, 794, 4

start = int(np.random.default_rng(42).integers(0, len(train_bin) - T_TEST))
x = torch.from_numpy(
    np.asarray(train_bin[start:start + T_TEST], dtype=np.int64)
).unsqueeze(0).to(DEVICE)

with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    full, _ = model_d(x)

    pl, states, plen = prefill_d(model_d, x[:, :T_PREFILL])
    assert plen == T_PREFILL

    got = [pl]
    for j in range(N_STEPS):
        lg, states = step_d(model_d, x[:, T_PREFILL + j:T_PREFILL + j + 1],
                            states, pos0=T_PREFILL + j)
        got.append(lg)
    got = torch.cat(got, dim=1)

ref = full[:, T_PREFILL - 1: T_PREFILL + N_STEPS]
assert got.shape == ref.shape, f"{got.shape} vs {ref.shape}"

diff  = (got.float() - ref.float()).abs()
agree = (got.argmax(-1) == ref.argmax(-1)).float().mean().item() * 100

W = model_d.blocks[0].mixer.window_size
klen = [s[0].shape[2] for s in states]

print(f"prompt {T_TEST} | window {W} | prefill {T_PREFILL}")
print(f"cache lengths per layer: {klen}")
print(f"positions compared : {T_PREFILL-1} .. {T_PREFILL+N_STEPS-1}")
print(f"max abs diff       : {diff.max().item():.6f}")
print(f"per-position max   : {[round(d, 4) for d in diff.amax(-1)[0].tolist()]}")
print(f"top-1 agreement    : {agree:.1f}%")
print("\nPASS" if diff.max().item() < 0.05 and agree == 100.0 else "\nFAIL")

prompt 800 | window 256 | prefill 794
cache lengths per layer: [256, 256, 798, 256, 256, 798, 256, 256, 798, 256, 256, 798]
positions compared : 793 .. 797
max abs diff       : 0.007812
per-position max   : [0.0039, 0.0078, 0.0078, 0.0078, 0.0078]
top-1 agreement    : 100.0%

PASS


In [20]:
import numpy as np
import torch

PAD_ID = int(tok.token_to_id("<|pad|>"))


@torch.no_grad()
def generate_batched_d(test_data, model, batch_size=32, max_new_tokens=350,
                       target_words=128, check_every=16, temperature=0.7,
                       top_p=0.9, top_k=50, repetition_penalty=1.1,
                       repetition_window=128, no_repeat_ngram_size=None,
                       verbose=True):
    model.eval()

    items = []
    for key, value in test_data.items():
        text = value["text"] if isinstance(value, dict) else value
        ids = tok.encode(text).ids
        items.append(dict(id=str(key), text=text, ids=ids, length=len(ids)))
    items.sort(key=lambda d: d["length"])

    real = sum(d["length"] for d in items)
    padded = sum(max(d["length"] for d in items[s:s + batch_size]) * len(items[s:s + batch_size])
                 for s in range(0, len(items), batch_size))
    if verbose:
        print(f"padding waste (sorted): {100*(padded-real)/padded:.2f}%")

    results, n_batches = {}, (len(items) + batch_size - 1) // batch_size

    for bi, start in enumerate(range(0, len(items), batch_size), 1):
        batch = items[start:start + batch_size]
        B = len(batch)
        max_len = max(d["length"] for d in batch)

        prompt = torch.full((B, max_len), PAD_ID, dtype=torch.long)
        for b, d in enumerate(batch):                       # LEFT pad
            prompt[b, max_len - d["length"]:] = torch.tensor(d["ids"], dtype=torch.long)
        prompt = prompt.to(DEVICE)

        history = prompt.clone()
        ngram_maps = [dict() for _ in range(B)]
        finished = torch.zeros(B, dtype=torch.bool, device=DEVICE)
        generated = [[] for _ in range(B)]

        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits, states, _ = prefill_d(model, prompt)
        abs_pos = max_len                    # absolute position of the next token

        for gstep in range(max_new_tokens):
            nxt = sample_next_token(
                logits[:, -1, :], history, ngram_maps,
                temperature=temperature, top_k=top_k, top_p=top_p,
                repetition_penalty=repetition_penalty,
                repetition_window=repetition_window,
                no_repeat_ngram_size=no_repeat_ngram_size)

            nxt = torch.where(finished.unsqueeze(1),
                              torch.full_like(nxt, PAD_ID), nxt)
            history = torch.cat([history, nxt], dim=1)

            if no_repeat_ngram_size:
                update_ngram_maps(ngram_maps, history, no_repeat_ngram_size)

            done_now = finished.tolist()
            for b, t in enumerate(nxt[:, 0].tolist()):
                if not done_now[b]:
                    generated[b].append(t)

            if (gstep + 1) % check_every == 0 or gstep + 1 == max_new_tokens:
                for b in range(B):
                    if not finished[b] and len(tok.decode(generated[b]).split()) >= target_words:
                        finished[b] = True
                if bool(finished.all()):
                    break

            if gstep + 1 < max_new_tokens:
                with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                    logits, states = step_d(model, nxt, states, pos0=abs_pos)
                abs_pos += 1

        for b, d in enumerate(batch):
            results[d["id"]] = dict(prompt=d["text"], generated_ids=generated[b])

        del states, history, prompt, logits
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        if verbose:
            print(f"  batch {bi}/{n_batches} | prompt_len {max_len} | "
                  f"gen {max(len(g) for g in generated)} tok", flush=True)

    return {str(k): results[str(k)] for k in test_data}        # restore order


# ---------------- 5-prompt smoke test ----------------
five = {str(k): test_data[k] for k in list(test_data)[:5]}
r = generate_batched_d(five, model_c, batch_size=5, max_new_tokens=350, verbose=False)
p, short = postprocess_predictions(r, five)
print(p["1"]["text"][:600])

AttributeError: 'GatedLinearRecurrence' object has no attribute 'n_heads'